# Text Classification

> Single-label and multi-label text classification with blurr

This notebook demonstrates how to fine-tune a HuggingFace transformer for text classification using blurr's clean, fastai-style API.

In [ ]:
#| default_exp text.classification

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from blurr.text.classification import TextClassificationLearner

## Single-Label Classification

### From a HuggingFace Dataset

The simplest way to get started — just pass a dataset name and model name:

In [ ]:
learn = TextClassificationLearner.from_hf(
    'imdb',
    model_name='prajjwal1/bert-tiny',  # Use a tiny model for demo
    text_col='text',
    label_col='label',
    max_samples=200,  # Limit for demo speed
    max_length=128,
    bs=16,
)

### Training

Use standard fastai training methods:

In [ ]:
# Find optimal learning rate
# learn.lr_find()

# Train for 1 epoch
learn.fit_one_cycle(1, 2e-5)

### Prediction

In [ ]:
# Single prediction
results = learn.blurr_predict('This movie was absolutely fantastic!')
print(results[0])

In [ ]:
# Batch prediction
results = learn.blurr_predict([
    'This movie was absolutely fantastic!',
    'Terrible waste of time, would not recommend.',
])
for r in results:
    print(f"Label: {r['label']}, Score: {r['score']:.4f}")

## From a DataFrame

You can also create a learner from a pandas DataFrame:

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'text': [
        'Great product!', 'Terrible quality.', 'Love it!', 'Waste of money.',
        'Highly recommended!', 'Worst purchase ever.', 'Amazing!', 'Awful product.',
        'Perfect fit!', 'Broken on arrival.', 'Superb quality.', 'Very disappointing.',
    ],
    'sentiment': ['positive'] * 3 + ['negative'] + ['positive'] + ['negative'] * 2 + ['positive'] * 2 + ['negative'] + ['positive'] + ['negative'],
})

learn = TextClassificationLearner.from_df(
    df,
    'prajjwal1/bert-tiny',
    text_col='text',
    label_col='sentiment',
    bs=4,
    max_length=64,
)
learn.fit_one_cycle(1, 2e-5)

## Multi-Label Classification

For multi-label tasks, pass a list of binary label columns:

In [ ]:
ml_df = pd.DataFrame({
    'text': [
        'You are so stupid!', 'Beautiful day today!', 'Go die, idiot!',
        'I love this!', 'Shut up moron!', 'Nice weather!',
        'You fool!', 'Great job!',
    ],
    'toxic': [1, 0, 1, 0, 1, 0, 1, 0],
    'insult': [1, 0, 1, 0, 1, 0, 1, 0],
})

learn = TextClassificationLearner.from_df(
    ml_df,
    'prajjwal1/bert-tiny',
    text_col='text',
    label_col=['toxic', 'insult'],
    is_multilabel=True,
    bs=4,
    max_length=64,
)
learn.fit_one_cycle(1, 2e-5)

## API Reference

### `TextClassificationLearner`

**Factory Methods:**
- `from_df(df, model_name, ...)` — Create from a pandas DataFrame
- `from_hf(dataset, model_name, ...)` — Create from a HuggingFace dataset
- `from_data(data, model_name, ...)` — Universal factory (auto-detects type)

**Prediction:**
- `blurr_predict(texts)` — Returns list of dicts with `label`, `score`, `probs`, `class_labels`

**Training:** All standard fastai `Learner` methods work:
- `fit_one_cycle()`, `fit()`, `lr_find()`, `freeze()`, `unfreeze()`, etc.